# Phase 2: Data Cleaning & Preprocessing

## Session 1: Data Type Correction

Fix TotalCharges type mismatch (string → numeric) identified in Phase 1 audit

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded: 7043 rows, 21 columns


In [3]:
# Inspect current data types
print("Current Data Types:")
print("=" * 60)
print(df.dtypes)
print("\n" + "=" * 60)

Current Data Types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object



In [3]:
# Check for type mismatches - focus on TotalCharges
print("Checking TotalCharges (should be numeric):")
print(f"Current type: {df['TotalCharges'].dtype}")
print(f"Sample values: {list(df['TotalCharges'].head())}")

# Check for empty/whitespace strings
empty_count = df['TotalCharges'].apply(lambda x: isinstance(x, str) and x.strip() == '').sum()
print(f"\nEmpty/whitespace strings: {empty_count}")

if empty_count > 0:
    print("\nRows with empty TotalCharges:")
    problematic = df[df['TotalCharges'].apply(lambda x: isinstance(x, str) and x.strip() == '')]
    print(problematic[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head())

Checking TotalCharges (should be numeric):
Current type: str
Sample values: ['29.85', '1889.5', '108.15', '1840.75', '151.65']

Empty/whitespace strings: 11

Rows with empty TotalCharges:
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             


## Type Conversion: TotalCharges

**Issue:** TotalCharges stored as string (blocks numeric operations)  
**Solution:** Convert to float64, replace empty strings with NaN using `pd.to_numeric(errors='coerce')`

In [4]:
# Convert TotalCharges: string -> float64
# Reason: Revenue column needs numeric type for calculations and modeling

print("Before conversion:")
print(f"Type: {df['TotalCharges'].dtype}")
print(f"NaN count: {df['TotalCharges'].isnull().sum()}")

# Convert using pd.to_numeric with errors='coerce'
# This converts empty strings to NaN automatically
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("\nAfter conversion:")
print(f"Type: {df['TotalCharges'].dtype}")
print(f"NaN count: {df['TotalCharges'].isnull().sum()}")
print(f"Sample values: {list(df['TotalCharges'].head())}")

Before conversion:
Type: str
NaN count: 0

After conversion:
Type: float64
NaN count: 11
Sample values: [29.85, 1889.5, 108.15, 1840.75, 151.65]


In [6]:
# Verify conversion success
print("Conversion Verification:")
print("=" * 60)

# Check all numeric columns are correct type
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']

for col in numeric_cols:
    dtype = df[col].dtype
    is_numeric = pd.api.types.is_numeric_dtype(df[col])
    status = "OK" if is_numeric else "ERROR"
    print(f"{col:<20} {str(dtype):<15} [{status}]")

print("\n" + "=" * 60)

Conversion Verification:
tenure               int64           [OK]
MonthlyCharges       float64         [OK]
TotalCharges         float64         [OK]
SeniorCitizen        int64           [OK]



In [7]:
# Final data type summary
print("Final Data Types After Correction:")
print("=" * 60)

type_summary = df.dtypes.value_counts()
print(type_summary)

print("\nMissing Values After Correction:")
print("=" * 60)
missing = df.isnull().sum()
missing = missing[missing > 0]

if len(missing) > 0:
    print(missing)
else:
    print("No missing values found")

print("\n" + "=" * 60)

Final Data Types After Correction:
str        17
int64       2
float64     2
Name: count, dtype: int64

Missing Values After Correction:
TotalCharges    11
dtype: int64



## Session 1 Summary

**Action:** Converted TotalCharges from string → float64  
**Result:** 11 empty strings became NaN (all tenure=0 new customers)  
**Verification:** All numeric columns now have correct data types

## Session 2: Missing Value Handling

Address 11 missing TotalCharges values (all from tenure=0 new signups)

In [8]:
# Identify all missing values in dataset
print("Missing Value Summary:")
print("=" * 60)

missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0]

if len(missing_summary) > 0:
    for col, count in missing_summary.items():
        pct = (count / len(df)) * 100
        print(f"{col:<20} {count:>5} rows ({pct:.2f}%)")
else:
    print("No missing values found")

print("\n" + "=" * 60)

Missing Value Summary:
TotalCharges            11 rows (0.16%)



In [9]:
# Investigate TotalCharges missing values
print("TotalCharges Missing Value Analysis:")
print("=" * 60)

missing_idx = df['TotalCharges'].isnull()
missing_rows = df[missing_idx]

print(f"\nTotal missing: {missing_idx.sum()} rows")
print("\nCharacteristics of missing rows:")
print(f"  - Tenure range: {missing_rows['tenure'].min()} to {missing_rows['tenure'].max()}")
print(f"  - MonthlyCharges range: ${missing_rows['MonthlyCharges'].min():.2f} to ${missing_rows['MonthlyCharges'].max():.2f}")
print(f"  - Churn rate: {(missing_rows['Churn'] == 'Yes').sum()}/{len(missing_rows)} ({(missing_rows['Churn'] == 'Yes').mean() * 100:.1f}%)")

print("\nSample rows with missing TotalCharges:")
print(missing_rows[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].head())

print("\n" + "=" * 60)

TotalCharges Missing Value Analysis:

Total missing: 11 rows

Characteristics of missing rows:
  - Tenure range: 0 to 0
  - MonthlyCharges range: $19.70 to $80.85
  - Churn rate: 0/11 (0.0%)

Sample rows with missing TotalCharges:
      customerID  tenure  MonthlyCharges  TotalCharges Churn
488   4472-LVYGI       0           52.55           NaN    No
753   3115-CZMZD       0           20.25           NaN    No
936   5709-LVOEQ       0           80.85           NaN    No
1082  4367-NUYAO       0           25.75           NaN    No
1340  1371-DWPAZ       0           56.05           NaN    No



## Missing Value Strategy

**Why Missing:** All 11 rows have tenure=0 (brand new customers, no cumulative charges yet)  
**Decision:** Impute with MonthlyCharges (realistic first-bill estimate)  
**Rationale:** Reflects expected billing for tenure=0 customers, preserves all 7,043 rows

In [5]:
# Handle missing TotalCharges values
# Strategy: Impute with MonthlyCharges for tenure=0 customers
# Reasoning: New customers haven't accumulated charges yet, but MonthlyCharges reflects expected first bill

print("Before Imputation:")
print(f"Missing TotalCharges: {df['TotalCharges'].isnull().sum()}")

# Create boolean mask for missing values
missing_mask = df['TotalCharges'].isnull()

# Impute: fill missing TotalCharges with MonthlyCharges
df.loc[missing_mask, 'TotalCharges'] = df.loc[missing_mask, 'MonthlyCharges']

print("\nAfter Imputation:")
print(f"Missing TotalCharges: {df['TotalCharges'].isnull().sum()}")

# Verify imputation worked correctly
print("\nImputed values (should match MonthlyCharges for tenure=0):")
imputed_customers = df[df['tenure'] == 0][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head(5)
print(imputed_customers)

Before Imputation:
Missing TotalCharges: 11

After Imputation:
Missing TotalCharges: 0

Imputed values (should match MonthlyCharges for tenure=0):
      customerID  tenure  MonthlyCharges  TotalCharges
488   4472-LVYGI       0           52.55         52.55
753   3115-CZMZD       0           20.25         20.25
936   5709-LVOEQ       0           80.85         80.85
1082  4367-NUYAO       0           25.75         25.75
1340  1371-DWPAZ       0           56.05         56.05


In [6]:
# Final verification: check entire dataset for missing values
print("Final Missing Value Check:")
print("=" * 60)

final_missing = df.isnull().sum()
final_missing_total = final_missing.sum()

if final_missing_total == 0:
    print("✓ No missing values in dataset")
else:
    print("⚠ Missing values still present:")
    print(final_missing[final_missing > 0])

print(f"\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n" + "=" * 60)

Final Missing Value Check:
✓ No missing values in dataset

Dataset shape: 7043 rows × 21 columns



## Session 2 Summary

**Action:** Imputed 11 missing TotalCharges with MonthlyCharges  
**Result:** Zero missing values, full 7,043 rows preserved  
**Note:** None of the 11 affected customers churned (100% retention)

## Session 3: Target & Binary Encoding

Encode Churn target and all Yes/No binary features (Yes=1, No=0)

In [7]:
# Identify all Yes/No binary columns
print("Binary Column Detection:")
print("=" * 60)

# Check unique values for each object column
binary_cols = []
for col in df.select_dtypes(include='object').columns:
    unique_vals = df[col].unique()
    if set(unique_vals) == {'Yes', 'No'}:
        binary_cols.append(col)
        print(f"{col:<20} → ['Yes', 'No']")

print(f"\nTotal binary columns found: {len(binary_cols)}")
print("\n" + "=" * 60)

Binary Column Detection:
Partner              → ['Yes', 'No']
Dependents           → ['Yes', 'No']
PhoneService         → ['Yes', 'No']
PaperlessBilling     → ['Yes', 'No']
Churn                → ['Yes', 'No']

Total binary columns found: 5



/var/folders/7f/mldk3sgj4550r3kh5dv86x140000gn/T/ipykernel_78132/109833199.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


## Encoding Strategy

**Target (Churn):** Create `churn_flag` (0=retained, 1=churned) - separate from features  
**Binary Features:** Encode Yes/No → 1/0, add `_flag` suffix for clarity  
**Keep:** Original columns temporarily for validation

In [8]:
# Encode target variable: Churn → churn_flag
# Reasoning: Separate target from features, clear binary encoding for modeling

print("Target Encoding:")
print("=" * 60)

# Create binary target: 1 = churned, 0 = retained
df['churn_flag'] = (df['Churn'] == 'Yes').astype(int)

print(f"Original Churn distribution:")
print(df['Churn'].value_counts())
print(f"\nEncoded churn_flag distribution:")
print(df['churn_flag'].value_counts())

# Verify encoding is correct
print("\nEncoding verification:")
print(df[['Churn', 'churn_flag']].drop_duplicates().sort_values('churn_flag'))

print("\n" + "=" * 60)

Target Encoding:
Original Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Encoded churn_flag distribution:
churn_flag
0    5174
1    1869
Name: count, dtype: int64

Encoding verification:
  Churn  churn_flag
0    No           0
2   Yes           1



In [9]:
# Encode all binary Yes/No features
# Reasoning: Consistent encoding (Yes=1, No=0) for all binary categorical features
# Exclude 'Churn' since we already created churn_flag as the target

print("Binary Feature Encoding:")
print("=" * 60)

# Get binary columns excluding Churn (already handled)
binary_features = [col for col in binary_cols if col != 'Churn']

print(f"Encoding {len(binary_features)} binary features:")
for col in binary_features:
    new_col_name = col.lower().replace(' ', '_') + '_flag'
    df[new_col_name] = (df[col] == 'Yes').astype(int)
    print(f"  {col:<20} → {new_col_name}")

print("\n" + "=" * 60)

Binary Feature Encoding:
Encoding 4 binary features:
  Partner              → partner_flag
  Dependents           → dependents_flag
  PhoneService         → phoneservice_flag
  PaperlessBilling     → paperlessbilling_flag



In [15]:
# Verify binary encoding correctness
print("Encoding Verification:")
print("=" * 60)

# Check a few binary features to ensure encoding is correct
sample_features = binary_features[:3] if len(binary_features) >= 3 else binary_features

for col in sample_features:
    flag_col = col.lower().replace(' ', '_') + '_flag'
    print(f"\n{col} → {flag_col}:")
    print(df[[col, flag_col]].drop_duplicates().sort_values(flag_col))

print("\n" + "=" * 60)

Encoding Verification:

Partner → partner_flag:
  Partner  partner_flag
1      No             0
0     Yes             1

Dependents → dependents_flag:
  Dependents  dependents_flag
0         No                0
6        Yes                1

PhoneService → phoneservice_flag:
  PhoneService  phoneservice_flag
0           No                  0
1          Yes                  1



In [16]:
# Check current dataset structure
print("Dataset Structure After Binary Encoding:")
print("=" * 60)

print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")

# Count encoded features
encoded_features = [col for col in df.columns if col.endswith('_flag')]
print(f"\nEncoded binary features: {len(encoded_features)}")
print(f"  - Target: churn_flag")
print(f"  - Features: {len(encoded_features) - 1}")

# Show data types
print(f"\nData type summary:")
print(df.dtypes.value_counts())

print("\n" + "=" * 60)

Dataset Structure After Binary Encoding:

Shape: 7043 rows × 26 columns

Encoded binary features: 5
  - Target: churn_flag
  - Features: 4

Data type summary:
str        17
int64       7
float64     2
Name: count, dtype: int64



## Session 3 Summary

**Action:** Created churn_flag target + encoded 4 binary features  
**Features:** Partner, Dependents, PhoneService, PaperlessBilling (all with `_flag` suffix)  
**Result:** Dataset expanded to 26 columns (kept originals for validation)

## Session 4: Categorical Encoding

Encode remaining multi-class categorical features (ordinal + one-hot)

In [10]:
# Identify remaining categorical columns (not yet encoded)
print("Remaining Categorical Columns:")
print("=" * 60)

# Get all string columns
str_cols = df.select_dtypes(include='object').columns.tolist()

# Exclude customerID (identifier, not feature) and already handled binary columns
exclude_cols = ['customerID', 'Churn', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
remaining_categorical = [col for col in str_cols if col not in exclude_cols]

print(f"\nTotal: {len(remaining_categorical)} columns to encode\n")

for col in remaining_categorical:
    unique_count = df[col].nunique()
    unique_vals = df[col].unique()
    print(f"{col:<20} | {unique_count} unique values")
    print(f"  → {list(unique_vals)}\n")

print("=" * 60)

Remaining Categorical Columns:

Total: 11 columns to encode

gender               | 2 unique values
  → ['Female', 'Male']

MultipleLines        | 3 unique values
  → ['No phone service', 'No', 'Yes']

InternetService      | 3 unique values
  → ['DSL', 'Fiber optic', 'No']

OnlineSecurity       | 3 unique values
  → ['No', 'Yes', 'No internet service']

OnlineBackup         | 3 unique values
  → ['Yes', 'No', 'No internet service']

DeviceProtection     | 3 unique values
  → ['No', 'Yes', 'No internet service']

TechSupport          | 3 unique values
  → ['No', 'Yes', 'No internet service']

StreamingTV          | 3 unique values
  → ['No', 'Yes', 'No internet service']

StreamingMovies      | 3 unique values
  → ['No', 'Yes', 'No internet service']

Contract             | 3 unique values
  → ['Month-to-month', 'One year', 'Two year']

PaymentMethod        | 4 unique values
  → ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']



/var/folders/7f/mldk3sgj4550r3kh5dv86x140000gn/T/ipykernel_78132/344833546.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include='object').columns.tolist()


## Encoding Strategy

**Ordinal:** Contract (0=month-to-month, 1=one year, 2=two year) - natural order by commitment  
**One-Hot:** gender, InternetService, MultipleLines, 7 service features, PaymentMethod  
**Settings:** Use `drop_first=True` to avoid multicollinearity (n-1 dummies)

In [11]:
# Ordinal Encoding: Contract
# Reasoning: Contract length has natural order - longer = more commitment
# Mapping: Month-to-month (0) < One year (1) < Two year (2)

print("Ordinal Encoding: Contract")
print("=" * 60)

print(f"Original Contract values:")
print(df['Contract'].value_counts().sort_index())

# Define ordinal mapping
contract_mapping = {
    'Month-to-month': 0,  # Least commitment
    'One year': 1,        # Medium commitment
    'Two year': 2         # Highest commitment
}

# Apply mapping
df['contract_ordinal'] = df['Contract'].map(contract_mapping)

print(f"\nEncoded contract_ordinal:")
print(df[['Contract', 'contract_ordinal']].drop_duplicates().sort_values('contract_ordinal'))
print(f"\nValue counts:")
print(df['contract_ordinal'].value_counts().sort_index())

print("\n" + "=" * 60)

Ordinal Encoding: Contract
Original Contract values:
Contract
Month-to-month    3875
One year          1473
Two year          1695
Name: count, dtype: int64

Encoded contract_ordinal:
          Contract  contract_ordinal
0   Month-to-month                 0
1         One year                 1
11        Two year                 2

Value counts:
contract_ordinal
0    3875
1    1473
2    1695
Name: count, dtype: int64



In [12]:
# One-Hot Encoding: gender (simple binary case)
# Reasoning: No natural order, Male vs Female
# Using drop_first=True to create 1 dummy variable (avoids multicollinearity)

print("One-Hot Encoding: gender")
print("=" * 60)

gender_encoded = pd.get_dummies(df['gender'], prefix='gender', drop_first=True, dtype=int)
df = pd.concat([df, gender_encoded], axis=1)

print(f"Encoded column created: {list(gender_encoded.columns)}")
print(f"\nInterpretation: gender_Male = 1 means Male, 0 means Female")
print(f"\nValue distribution:")
print(df['gender_Male'].value_counts())

print("\n" + "=" * 60)

One-Hot Encoding: gender
Encoded column created: ['gender_Male']

Interpretation: gender_Male = 1 means Male, 0 means Female

Value distribution:
gender_Male
1    3555
0    3488
Name: count, dtype: int64



In [13]:
# One-Hot Encoding: InternetService
# Reasoning: 3 categories (DSL, Fiber optic, No) - no natural order
# Drop_first=True to avoid dummy variable trap

print("One-Hot Encoding: InternetService")
print("=" * 60)

internet_encoded = pd.get_dummies(df['InternetService'], prefix='internet', drop_first=True, dtype=int)
df = pd.concat([df, internet_encoded], axis=1)

print(f"Encoded columns: {list(internet_encoded.columns)}")
print(f"\nOriginal distribution:")
print(df['InternetService'].value_counts())
print(f"\nEncoded columns created:")
for col in internet_encoded.columns:
    print(f"  {col}: {df[col].sum()} customers")

print("\n" + "=" * 60)

One-Hot Encoding: InternetService
Encoded columns: ['internet_Fiber optic', 'internet_No']

Original distribution:
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

Encoded columns created:
  internet_Fiber optic: 3096 customers
  internet_No: 1526 customers



In [14]:
# One-Hot Encoding: Phone & Internet service features
# Reasoning: All have 3 categories (No, Yes, No service) - no natural order
# Batch encoding for: MultipleLines, OnlineSecurity, OnlineBackup, DeviceProtection, 
#                     TechSupport, StreamingTV, StreamingMovies

print("One-Hot Encoding: Service Features")
print("=" * 60)

service_features = [
    'MultipleLines',
    'OnlineSecurity', 
    'OnlineBackup', 
    'DeviceProtection',
    'TechSupport', 
    'StreamingTV', 
    'StreamingMovies'
]

print(f"Encoding {len(service_features)} service features:\n")

for feature in service_features:
    # Create one-hot encoded columns
    encoded = pd.get_dummies(df[feature], prefix=feature.lower(), drop_first=True, dtype=int)
    df = pd.concat([df, encoded], axis=1)
    
    print(f"{feature:<20} → {list(encoded.columns)}")

print("\n" + "=" * 60)

One-Hot Encoding: Service Features
Encoding 7 service features:

MultipleLines        → ['multiplelines_No phone service', 'multiplelines_Yes']
OnlineSecurity       → ['onlinesecurity_No internet service', 'onlinesecurity_Yes']
OnlineBackup         → ['onlinebackup_No internet service', 'onlinebackup_Yes']
DeviceProtection     → ['deviceprotection_No internet service', 'deviceprotection_Yes']
TechSupport          → ['techsupport_No internet service', 'techsupport_Yes']
StreamingTV          → ['streamingtv_No internet service', 'streamingtv_Yes']
StreamingMovies      → ['streamingmovies_No internet service', 'streamingmovies_Yes']



In [15]:
# One-Hot Encoding: PaymentMethod
# Reasoning: 4 payment types - no natural order
# Drop_first=True to create 3 dummy variables

print("One-Hot Encoding: PaymentMethod")
print("=" * 60)

payment_encoded = pd.get_dummies(df['PaymentMethod'], prefix='payment', drop_first=True, dtype=int)
df = pd.concat([df, payment_encoded], axis=1)

print(f"Encoded columns: {list(payment_encoded.columns)}")
print(f"\nOriginal distribution:")
print(df['PaymentMethod'].value_counts())
print(f"\nEncoded columns created:")
for col in payment_encoded.columns:
    print(f"  {col}: {df[col].sum()} customers")

print("\n" + "=" * 60)

One-Hot Encoding: PaymentMethod
Encoded columns: ['payment_Credit card (automatic)', 'payment_Electronic check', 'payment_Mailed check']

Original distribution:
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64

Encoded columns created:
  payment_Credit card (automatic): 1522 customers
  payment_Electronic check: 2365 customers
  payment_Mailed check: 1612 customers



In [23]:
# Verify encoding completeness
print("Encoding Verification:")
print("=" * 60)

# Check for remaining string columns (exclude customerID)
remaining_str = df.select_dtypes(include='object').columns.tolist()
print(f"\nRemaining string columns: {len(remaining_str)}")
print(f"  {remaining_str}")

# Count encoded features
encoded_cols = [col for col in df.columns if col not in df.select_dtypes(include='object').columns]
encoded_cols = [col for col in encoded_cols if col not in ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']]

print(f"\nTotal encoded features: {len(encoded_cols)}")
print(f"  - Binary flags: {len([c for c in encoded_cols if c.endswith('_flag')])}")
print(f"  - Ordinal: {len([c for c in encoded_cols if 'ordinal' in c])}")
print(f"  - One-hot: {len(encoded_cols) - len([c for c in encoded_cols if c.endswith('_flag')]) - len([c for c in encoded_cols if 'ordinal' in c])}")

print(f"\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

print("\n" + "=" * 60)

Encoding Verification:

Remaining string columns: 17
  ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']

Total encoded features: 26
  - Binary flags: 5
  - Ordinal: 1
  - One-hot: 20

Dataset shape: 7043 rows × 47 columns



/var/folders/7f/mldk3sgj4550r3kh5dv86x140000gn/T/ipykernel_77289/356431344.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  remaining_str = df.select_dtypes(include='object').columns.tolist()
/var/folders/7f/mldk3sgj4550r3kh5dv86x140000gn/T/ipykernel_77289/356431344.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See 

In [16]:
# Drop original categorical columns (keep customerID for reference)
# Reasoning: All information now captured in encoded features

print("Dropping Original Categorical Columns:")
print("=" * 60)

# List of original categorical columns to drop
cols_to_drop = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'Churn'
]

print(f"Dropping {len(cols_to_drop)} original columns:")
for col in cols_to_drop:
    print(f"  - {col}")

df_clean = df.drop(columns=cols_to_drop)

print(f"\nDataset shape after dropping:")
print(f"  Before: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  After:  {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")

print("\n" + "=" * 60)

Dropping Original Categorical Columns:
Dropping 16 original columns:
  - gender
  - Partner
  - Dependents
  - PhoneService
  - MultipleLines
  - InternetService
  - OnlineSecurity
  - OnlineBackup
  - DeviceProtection
  - TechSupport
  - StreamingTV
  - StreamingMovies
  - Contract
  - PaperlessBilling
  - PaymentMethod
  - Churn

Dataset shape after dropping:
  Before: 7043 rows × 47 columns
  After:  7043 rows × 31 columns



In [17]:
# Final clean dataset summary
print("Final Clean Dataset Summary:")
print("=" * 60)

print(f"\nShape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")

print(f"\nData types:")
print(df_clean.dtypes.value_counts())

print(f"\nColumn categories:")
print(f"  - Customer ID: 1 (customerID)")
print(f"  - Target: 1 (churn_flag)")
print(f"  - Numeric features: 4 (tenure, MonthlyCharges, TotalCharges, SeniorCitizen)")
print(f"  - Binary flags: {len([c for c in df_clean.columns if c.endswith('_flag') and c != 'churn_flag'])}")
print(f"  - Ordinal features: {len([c for c in df_clean.columns if 'ordinal' in c])}")
print(f"  - One-hot features: {df_clean.shape[1] - 1 - 4 - len([c for c in df_clean.columns if c.endswith('_flag')]) - len([c for c in df_clean.columns if 'ordinal' in c])}")

print(f"\nMissing values: {df_clean.isnull().sum().sum()}")

print(f"\nSample columns:")
print(list(df_clean.columns[:10]))

print("\n" + "=" * 60)

Final Clean Dataset Summary:

Shape: 7043 rows × 31 columns

Data types:
int64      28
float64     2
str         1
Name: count, dtype: int64

Column categories:
  - Customer ID: 1 (customerID)
  - Target: 1 (churn_flag)
  - Numeric features: 4 (tenure, MonthlyCharges, TotalCharges, SeniorCitizen)
  - Binary flags: 4
  - Ordinal features: 1
  - One-hot features: 20

Missing values: 0

Sample columns:
['customerID', 'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'churn_flag', 'partner_flag', 'dependents_flag', 'phoneservice_flag', 'paperlessbilling_flag']



## Session 4 Summary

**Ordinal:** Contract (0-2 by commitment level)  
**One-Hot:** 9 features encoded with drop_first=True  
**Cleanup:** Dropped 16 original categorical columns (kept customerID)  
**Result:** 7,043 rows × 31 columns, all numeric, zero missing values

## Data Cleaning Complete: Overall Summary

**Session 1:** Fixed TotalCharges type (string → float64)  
**Session 2:** Imputed 11 missing values (tenure=0 customers)  
**Session 3:** Encoded target + 4 binary features  
**Session 4:** Ordinal (Contract) + one-hot (9 features), dropped originals  

**Final Dataset:** 7,043 rows × 31 columns (1 target + 30 features, all numeric, 0 missing)  
**Status:** ✓ Ready for exploratory analysis and modeling

## Session 5: Feature Matrix Preparation

Prepare model-ready data: separate features from target, scale numeric columns appropriately

In [18]:
# Step 1: Separate Feature Matrix (X) and Target (y)
# Reasoning: Models need features and target as separate inputs

print("Separating Features and Target:")
print("=" * 60)

# Exclude customerID (not a feature) and churn_flag (target)
feature_cols = [col for col in df_clean.columns if col not in ['customerID', 'churn_flag']]

X = df_clean[feature_cols].copy()
y = df_clean['churn_flag'].copy()

print(f"Feature matrix (X): {X.shape[0]} rows × {X.shape[1]} columns")
print(f"Target variable (y): {y.shape[0]} values")
print(f"\nTarget distribution:")
print(f"  No churn (0): {(y == 0).sum()} customers ({(y == 0).mean() * 100:.1f}%)")
print(f"  Churned (1):  {(y == 1).sum()} customers ({(y == 1).mean() * 100:.1f}%)")

print("\n" + "=" * 60)

Separating Features and Target:
Feature matrix (X): 7043 rows × 29 columns
Target variable (y): 7043 values

Target distribution:
  No churn (0): 5174 customers (73.5%)
  Churned (1):  1869 customers (26.5%)



In [19]:
# Step 2: Identify columns that need scaling
# Reasoning: Different feature types need different treatment

print("Feature Analysis for Scaling:")
print("=" * 60)

# Analyze all features
print("\nFeature types in X:")

# Get all columns
all_features = X.columns.tolist()

# Identify continuous numeric columns (need scaling)
continuous_numeric = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

# Identify binary/one-hot columns (NO scaling needed - already 0/1)
binary_onehot = [col for col in all_features if col not in continuous_numeric]

print(f"\n1. Continuous numeric features ({len(continuous_numeric)}):")
for col in continuous_numeric:
    min_val = X[col].min()
    max_val = X[col].max()
    mean_val = X[col].mean()
    print(f"   {col:<20} | Range: [{min_val:.1f}, {max_val:.1f}] | Mean: {mean_val:.1f}")

print(f"\n2. Binary/One-hot features ({len(binary_onehot)}):")
print(f"   {binary_onehot[:5]}...")
print(f"   (All are 0/1 values - NO scaling needed)")

print("\n" + "=" * 60)

Feature Analysis for Scaling:

Feature types in X:

1. Continuous numeric features (4):
   SeniorCitizen        | Range: [0.0, 1.0] | Mean: 0.2
   tenure               | Range: [0.0, 72.0] | Mean: 32.4
   MonthlyCharges       | Range: [18.2, 118.8] | Mean: 64.8
   TotalCharges         | Range: [18.8, 8684.8] | Mean: 2279.8

2. Binary/One-hot features (25):
   ['partner_flag', 'dependents_flag', 'phoneservice_flag', 'paperlessbilling_flag', 'contract_ordinal']...
   (All are 0/1 values - NO scaling needed)



## Scaling Strategy

**Scale:** Only continuous numeric features (SeniorCitizen, tenure, MonthlyCharges, TotalCharges)  
**Don't Scale:** Binary flags and one-hot encoded features (already 0/1)  
**Method:** StandardScaler (mean=0, std=1) - good for logistic regression  
**Preserve:** Column names after scaling for interpretability

In [20]:
# Step 3: Apply scaling to continuous features only
# Reasoning: Logistic regression needs scaled features, but tree models don't require it
# Strategy: Scale continuous columns, keep binary/one-hot as is

from sklearn.preprocessing import StandardScaler

print("Applying StandardScaler to Continuous Features:")
print("=" * 60)

# Initialize scaler
scaler = StandardScaler()

# Scale only continuous numeric columns
X_scaled_continuous = scaler.fit_transform(X[continuous_numeric])

# Convert back to DataFrame with original column names
X_scaled_continuous_df = pd.DataFrame(
    X_scaled_continuous, 
    columns=continuous_numeric,
    index=X.index
)

# Combine scaled continuous features with unscaled binary/one-hot features
X_scaled = pd.concat([X_scaled_continuous_df, X[binary_onehot]], axis=1)

# Reorder columns to match original X
X_scaled = X_scaled[all_features]

print(f"Scaled features: {list(continuous_numeric)}")
print(f"\nUnscaled features: {len(binary_onehot)} binary/one-hot columns")

print(f"\nFinal feature matrix shape: {X_scaled.shape[0]} rows × {X_scaled.shape[1]} columns")

print("\n" + "=" * 60)

Applying StandardScaler to Continuous Features:
Scaled features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Unscaled features: 25 binary/one-hot columns

Final feature matrix shape: 7043 rows × 29 columns



In [21]:
# Step 4: Verify scaling results
print("Scaling Verification:")
print("=" * 60)

# Check scaled continuous features (should have mean≈0, std≈1)
print("\nScaled continuous features (mean ≈ 0, std ≈ 1):")
for col in continuous_numeric:
    mean_val = X_scaled[col].mean()
    std_val = X_scaled[col].std()
    print(f"  {col:<20} | Mean: {mean_val:.6f} | Std: {std_val:.6f}")

# Check binary/one-hot features (should still be 0/1)
print(f"\nBinary/one-hot features (still 0/1):")
sample_binary = binary_onehot[:3]
for col in sample_binary:
    unique_vals = sorted(X_scaled[col].unique())
    print(f"  {col:<40} | Unique values: {unique_vals}")

# Verify column names preserved
print(f"\nColumn names preserved: {list(X_scaled.columns) == all_features}")

print("\n" + "=" * 60)

Scaling Verification:

Scaled continuous features (mean ≈ 0, std ≈ 1):
  SeniorCitizen        | Mean: -0.000000 | Std: 1.000071
  tenure               | Mean: -0.000000 | Std: 1.000071
  MonthlyCharges       | Mean: -0.000000 | Std: 1.000071
  TotalCharges         | Mean: -0.000000 | Std: 1.000071

Binary/one-hot features (still 0/1):
  partner_flag                             | Unique values: [np.int64(0), np.int64(1)]
  dependents_flag                          | Unique values: [np.int64(0), np.int64(1)]
  phoneservice_flag                        | Unique values: [np.int64(0), np.int64(1)]

Column names preserved: True



In [22]:
# Final Summary: Model-Ready Data
print("Model-Ready Dataset Summary:")
print("=" * 60)

print(f"\nFeature Matrix (X_scaled):")
print(f"  Shape: {X_scaled.shape[0]} rows × {X_scaled.shape[1]} columns")
print(f"  Data type: All numeric")
print(f"  Missing values: {X_scaled.isnull().sum().sum()}")
print(f"  Scaled features: {len(continuous_numeric)} continuous variables")
print(f"  Original scale: {len(binary_onehot)} binary/one-hot variables")

print(f"\nTarget Variable (y):")
print(f"  Shape: {len(y)} values")
print(f"  Type: Binary (0=No churn, 1=Churned)")
print(f"  Class balance: {(y==1).sum()}/{len(y)} churned ({(y==1).mean()*100:.1f}%)")

print(f"\nReadiness Check:")
print(f"  ✓ Features and target separated")
print(f"  ✓ Continuous features scaled (logistic regression ready)")
print(f"  ✓ Binary features preserved (tree model compatible)")
print(f"  ✓ Column names maintained")
print(f"  ✓ No missing values")
print(f"  ✓ All features numeric")

print("\n" + "=" * 60)

Model-Ready Dataset Summary:

Feature Matrix (X_scaled):
  Shape: 7043 rows × 29 columns
  Data type: All numeric
  Missing values: 0
  Scaled features: 4 continuous variables
  Original scale: 25 binary/one-hot variables

Target Variable (y):
  Shape: 7043 values
  Type: Binary (0=No churn, 1=Churned)
  Class balance: 1869/7043 churned (26.5%)

Readiness Check:
  ✓ Features and target separated
  ✓ Continuous features scaled (logistic regression ready)
  ✓ Binary features preserved (tree model compatible)
  ✓ Column names maintained
  ✓ No missing values
  ✓ All features numeric



## Session 5 Summary

### What We Did:
1. **Split the Data into Two Parts**
   - X: 7,043 customers × 29 features (all customer information)
   - y: 7,043 labels showing who left (26.5% churned, 73.5% stayed)

2. **Identified Numbers That Needed Adjustment**
   - Found 4 columns with different scales:
     - tenure: 0 to 72 months
     - MonthlyCharges: $18 to $119
     - TotalCharges: $19 to $8,685
     - SeniorCitizen: 0 or 1
   - Left 25 other columns alone (already simple 0/1 values)

3. **Applied Smart Scaling** 
   - Used StandardScaler to adjust the 4 continuous numbers
   - Now all scaled features have: mean ≈ 0, standard deviation ≈ 1
   - This puts all numbers on the same "playing field" for fair comparison

### Why This Matters:
- Imagine comparing apples (MonthlyCharges: ~$65) with elephants (TotalCharges: ~$2,280)
- Without scaling, the bigger numbers would dominate the model's decisions
- With scaling, each feature gets equal importance based on its patterns, not its size

### Final Result:
✓ **X_scaled** = 7,043 rows × 29 features (ready for any model)  
✓ **y** = 7,043 labels (who churned)  
✓ Works for both logistic regression (needs scaling) AND tree models (doesn't need it)